In [1]:
from mlxtend.frequent_patterns import apriori
from mlxtend.frequent_patterns import association_rules
from mlxtend.preprocessing.transactionencoder import TransactionEncoder
import csv
import os
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt

# 1

In [2]:
# 读取原始数据
movies = pd.read_csv('movie.csv')
movies.head(10)  

,电影名称,豆瓣评论数,豆瓣评分,上映日期,主演,制片国家/地区,又名,导演,片长,类型,编剧,语言,r5,r4,r3,r2,r1
0,坏孩子的秋天,263.0,7.5,暂无,肖政 / 施大生 / 贾媛媛 / 王静,中国大陆,NaN,邓科,90,剧情 / 儿童,陆长河,汉语普通话,0.220,0.424,0.280,0.059,0.017
1,黑猫大旅社 黑貓大旅社,514.0,6.8,不详,陆奕静 / 老鄧 （夏靖庭 ） / 蔡振南,台湾,Hotel Black Cat,徐丽雯,112,剧情,徐丽雯,台湾国语,0.102,0.332,0.453,0.091,0.022
2,追捕,NaN,NaN,2018-02(香港),张涵予 / 戚薇 / 福山雅治 / 河智苑 / 国村隼 / 池内博之 / 樱庭奈奈美 / ...,香港,Manhunt,吴宇森,NaN,剧情 / 动作,吴宇森 / 阮世生 / 西村寿行,NaN,0.000,0.000,0.000,0.000,0.000
3,无问西东,NaN,NaN,2018(中国大陆),章子怡 / 张震 / 黄晓明 / 王力宏 / 陈楚生 / 韩童生 / 王盛德 / 姚晨 /...,中国大陆,Forever Young,李芳芳,NaN,剧情 / 爱情 / 战争,李芳芳,汉语普通话,0.000,0.000,0.000,0.000,0.000
4,绣春刀II：修罗战场,NaN,NaN,2017-07(中国大陆),张震 / 杨幂 / 张译 / 雷佳音 / 辛芷蕾 / 李媛,中国大陆,绣春刀前传 / 绣春刀：修罗场 / 绣春刀2：修罗战场,路阳,NaN,动作 / 武侠 / 古装,陈舒,汉语普通话,0.000,0.000,0.000,0.000,0.000
5,闺蜜2,NaN,NaN,2017-04-30(中国大陆),陈意涵 / 张钧甯 / 薛凯琪,中国大陆,闺蜜2：单挑越南黑帮,黄真真,NaN,爱情,孙艳杰,汉语普通话,0.000,0.000,0.000,0.000,0.000
6,记忆大师,NaN,NaN,2017-04-28(中国大陆),黄渤 / 徐静蕾 / 段奕宏 / 杨子姗 / 许玮甯,中国大陆,记忆战 / 催眠大师2 / Battle of Memories,陈正道,NaN,悬疑 / 犯罪,任鹏,汉语普通话,0.000,0.000,0.000,0.000,0.000
7,我心雀跃,209.0,6.7,2017-04(中国大陆) / 2016-06-14(上海国际电影节),孙伊涵 / 宋宁 / 周楚楚 / 修健 / 杜双宇 / 刘锐 / 池韵 / 刘北妍 / 任运杰,中国大陆,My Heart Leaps Up,刘紫微,95分钟,剧情,刘紫微,汉语普通话,0.086,0.345,0.432,0.115,0.022
8,美好的意外,NaN,NaN,2017-03-17(中国大陆),桂纶镁 / 王景春 / 欧阳娜娜 / 王元也 / 陈坤,中国大陆,美满人间 / 美好人生 / Beautiful Accident,何蔚庭,NaN,喜剧 / 奇幻,哈智超 / 周佳琳,汉语普通话,0.000,0.000,0.000,0.000,0.000
9,嫌疑人X的献身,NaN,NaN,2017-03(中国大陆),王凯 / 柳岩 / 张鲁一 / 林心如,中国大陆,Suspect X,苏有朋,NaN,剧情 / 悬疑 / 犯罪,东野圭吾,汉语普通话,0.000,0.000,0.000,0.000,0.000


In [5]:
mov=movies[['电影名称','类型']]
mov.head()

,电影名称,类型
0,坏孩子的秋天,剧情 / 儿童
1,黑猫大旅社 黑貓大旅社,剧情
2,追捕,剧情 / 动作
3,无问西东,剧情 / 爱情 / 战争
4,绣春刀II：修罗战场,动作 / 武侠 / 古装


In [9]:
# 第一步，将原始数据转化为标准格式
movies_standard = mov.drop('类型', axis=1).join(mov['类型'].replace(r'\s*/\s*', '|', regex=True).str.strip().str.get_dummies())

In [10]:
#movies_standard = mov.drop('类型', 1).join(mov['类型'].str.get_dummies())
print(movies_standard.head(10))
# 一共包含 250部电影，一共有29种不同的电影类型（有2列是ID和电影名）
print(movies_standard.shape)  # (250, 30)

          电影名称  News  传记  儿童  冒险  剧情  动作  动画  历史  古装  ...  短片  科幻  纪录片  脱口秀  \
0       坏孩子的秋天     0   0   1   0   1   0   0   0   0  ...   0   0    0    0   
1  黑猫大旅社 黑貓大旅社     0   0   0   0   1   0   0   0   0  ...   0   0    0    0   
2           追捕     0   0   0   0   1   1   0   0   0  ...   0   0    0    0   
3         无问西东     0   0   0   0   1   0   0   0   0  ...   0   0    0    0   
4   绣春刀II：修罗战场     0   0   0   0   0   1   0   0   1  ...   0   0    0    0   
5          闺蜜2     0   0   0   0   0   0   0   0   0  ...   0   0    0    0   
6         记忆大师     0   0   0   0   0   0   0   0   0  ...   0   0    0    0   
7         我心雀跃     0   0   0   0   1   0   0   0   0  ...   0   0    0    0   
8        美好的意外     0   0   0   0   0   0   0   0   0  ...   0   0    0    0   
9      嫌疑人X的献身     0   0   0   0   1   0   0   0   0  ...   0   0    0    0   

   舞台艺术  西部  运动  音乐  鬼怪  黑色电影  
0     0   0   0   0   0     0  
1     0   0   0   0   0     0  
2     0   0   0   0   0     0  
3 

In [11]:
movies_standard.columns

Index(['电影名称', 'News', '传记', '儿童', '冒险', '剧情', '动作', '动画', '历史', '古装', '同性',
       '喜剧', '奇幻', '家庭', '恐怖', '悬疑', '情色', '惊悚', '戏曲', '战争', '歌舞', '武侠', '灾难',
       '爱情', '犯罪', '真人秀', '短片', '科幻', '纪录片', '脱口秀', '舞台艺术', '西部', '运动', '音乐',
       '鬼怪', '黑色电影'],
      dtype='object')

In [19]:
# 利用mlxtend提供的apriori算法函数得到频繁项集，其中设置最小支持度为0.06
#movies_standard.set_index(['电影名称'], inplace=True)
frequent_item_sets = apriori(movies_standard, min_support=0.06, use_colnames=True)
print(frequent_item_sets)

    support  itemsets
0  0.422873      (剧情)
1  0.136345      (动作)
2  0.243793      (喜剧)
3  0.095645      (悬疑)
4  0.080586      (惊悚)
5  0.235246      (爱情)
6  0.089540  (剧情, 爱情)
7  0.096052  (喜剧, 爱情)


D:\soft\python\Lib\site-packages\mlxtend\frequent_patterns\fpcommon.py:113: DeprecationWarning: DataFrames with non-bool types result in worse computationalperformance and their support might be discontinued in the future.Please use a DataFrame with bool type
  DeprecationWarning,


In [20]:
frequent_item_sets.sort_values(by='support',ascending=False)

,support,itemsets
0,0.422873,(剧情)
2,0.243793,(喜剧)
5,0.235246,(爱情)
1,0.136345,(动作)
7,0.096052,"(喜剧, 爱情)"
3,0.095645,(悬疑)
6,0.089540,"(剧情, 爱情)"
4,0.080586,(惊悚)


In [22]:
# 计算规则，并设置提升度阈值为 1.25 （返回的是各个指标的数值，可以按照按兴趣的指标排序观察，但具体解释还得参考实际数据的含义）
rules = association_rules(frequent_item_sets, metric='lift', min_threshold=1)
rules

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,leverage,conviction,zhangs_metric
0,(喜剧),(爱情),0.243793,0.235246,0.096052,0.393990,1.674798,0.038701,1.261949,0.532808
1,(爱情),(喜剧),0.235246,0.243793,0.096052,0.408304,1.674798,0.038701,1.278034,0.526853


In [25]:
rules = association_rules(frequent_item_sets, metric='confidence', min_threshold=0.4)
rules

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,leverage,conviction,zhangs_metric
0,(爱情),(喜剧),0.235246,0.243793,0.096052,0.408304,1.674798,0.038701,1.278034,0.526853


In [3]:
mov=movies[['电影名称','主演']]
mov.head()

,电影名称,主演
0,坏孩子的秋天,肖政 / 施大生 / 贾媛媛 / 王静
1,黑猫大旅社 黑貓大旅社,陆奕静 / 老鄧 （夏靖庭 ） / 蔡振南
2,追捕,张涵予 / 戚薇 / 福山雅治 / 河智苑 / 国村隼 / 池内博之 / 樱庭奈奈美 / ...
3,无问西东,章子怡 / 张震 / 黄晓明 / 王力宏 / 陈楚生 / 韩童生 / 王盛德 / 姚晨 /...
4,绣春刀II：修罗战场,张震 / 杨幂 / 张译 / 雷佳音 / 辛芷蕾 / 李媛


In [4]:
# 第一步，将原始数据转化为标准格式
movies_standard = mov.drop('主演', axis=1).join(mov['主演'].replace(r'\s*/\s*', '|', regex=True).str.strip().str.get_dummies())

In [5]:
#movies_standard = mov.drop('类型', 1).join(mov['类型'].str.get_dummies())
print(movies_standard.head(10))
# 一共包含 250部电影，一共有29种不同的电影类型（有2列是ID和电影名）
print(movies_standard.shape)  # (250, 30)

          电影名称  3000  Aaron Shang  Adila Shakir  Adinia Wirasti（阿迪妮．威拉丝缇）  \
0       坏孩子的秋天     0            0             0                         0   
1  黑猫大旅社 黑貓大旅社     0            0             0                         0   
2           追捕     0            0             0                         0   
3         无问西东     0            0             0                         0   
4   绣春刀II：修罗战场     0            0             0                         0   
5          闺蜜2     0            0             0                         0   
6         记忆大师     0            0             0                         0   
7         我心雀跃     0            0             0                         0   
8        美好的意外     0            0             0                         0   
9      嫌疑人X的献身     0            0             0                         0   

   Amanda Baden  Amleto Voltolina  Andrea Pennacchi  Andrew Susay  \
0             0                 0                 0             0   
1             

In [7]:
# 利用mlxtend提供的apriori算法函数得到频繁项集，其中设置最小支持度为0.06
movies_standard.set_index(['电影名称'], inplace=True)
frequent_item_sets = apriori(movies_standard, min_support=0.06, use_colnames=True)
print(frequent_item_sets)

Empty DataFrame
Columns: [support, itemsets]
Index: []


D:\soft\python\Lib\site-packages\mlxtend\frequent_patterns\fpcommon.py:113: DeprecationWarning: DataFrames with non-bool types result in worse computationalperformance and their support might be discontinued in the future.Please use a DataFrame with bool type
  DeprecationWarning,


In [8]:
frequent_item_sets = apriori(movies_standard, min_support=0.01, use_colnames=True)
print(frequent_item_sets)

    support itemsets
0  0.012617    (任达华)
1  0.012210     (刘桦)
2  0.010175    (古天乐)
3  0.013024    (曾志伟)
4  0.010989    (杜海涛)
5  0.015873     (林雪)


D:\soft\python\Lib\site-packages\mlxtend\frequent_patterns\fpcommon.py:113: DeprecationWarning: DataFrames with non-bool types result in worse computationalperformance and their support might be discontinued in the future.Please use a DataFrame with bool type
  DeprecationWarning,


In [2]:
# 读取原始数据
movies = pd.read_csv('movie2.csv')
movies.head(10)  

,Unnamed: 0,类型,主演,地区,导演,特色,评分,电影名
0,0,剧情,徐峥|王传君|周一围|谭卓|章宇,中国大陆,文牧野,经典,8.9,我不是药神
1,1,剧情,冯小刚|许晴|张涵予|刘桦|李易峰,中国大陆,管虎,经典,7.8,老炮儿
2,2,剧情,王宝强|刘昊然|肖央|刘承羽|尚语贤,中国大陆,陈思诚,经典,6.7,唐人街探案2
3,3,剧情,任素汐|大力|刘帅良|裴魁山|阿如那,中国大陆,周申|刘露,经典,8.3,驴得水
4,4,剧情,徐峥|王宝强|李曼|李小璐|左小青,中国大陆,叶伟民,经典,7.5,人在囧途
5,5,剧情,徐峥|黄渤|余男|多布杰|王双宝,中国大陆,宁浩,经典,8.1,无人区
6,6,剧情,姜文|香川照之|袁丁|姜宏波|丛志军,中国大陆,姜文,经典,9.2,鬼子来了
7,7,剧情,章子怡|黄晓明|张震|王力宏|陈楚生,中国大陆,李芳芳,经典,7.6,无问西东
8,8,剧情,彭于晏|廖凡|姜文|周韵|许晴,中国大陆,姜文,经典,7.1,邪不压正
9,9,剧情,肖央|王太利|韩秋池|于蓓蓓,中国大陆,肖央,经典,8.5,11度青春之


In [32]:
movi2=movies[movies['地区']=='中国大陆']

In [33]:
movi2.shape

(9071, 7)

In [38]:
movi2.to_csv('movie2.csv')

In [4]:
mov=movies[['电影名','主演']]
mov.head()

,电影名,主演
0,我不是药神,徐峥|王传君|周一围|谭卓|章宇
1,老炮儿,冯小刚|许晴|张涵予|刘桦|李易峰
2,唐人街探案2,王宝强|刘昊然|肖央|刘承羽|尚语贤
3,驴得水,任素汐|大力|刘帅良|裴魁山|阿如那
4,人在囧途,徐峥|王宝强|李曼|李小璐|左小青


In [5]:
# 第一步，将原始数据转化为标准格式
movies_standard = mov.drop('主演', 1).join(mov['主演'].str.get_dummies())

D:\soft\python\Lib\site-packages\ipykernel_launcher.py:2: FutureWarning: In a future version of pandas all arguments of DataFrame.drop except for the argument 'labels' will be keyword-only
  


In [37]:
#movies_standard = mov.drop('类型', 1).join(mov['类型'].str.get_dummies())
print(movies_standard.head(10))
# 一共包含 250部电影，一共有29种不同的电影类型（有2列是ID和电影名）
print(movies_standard.shape)  # (250, 30)

      电影名  14  AYO哟  Abby  Aron  A大调  BY2  Bailey  Bernardo Loureiro  Bin  \
0   我不是药神   0     0     0     0    0    0       0                  0    0   
1     老炮儿   0     0     0     0    0    0       0                  0    0   
2  唐人街探案2   0     0     0     0    0    0       0                  0    0   
3     驴得水   0     0     0     0    0    0       0                  0    0   
4    人在囧途   0     0     0     0    0    0       0                  0    0   
5     无人区   0     0     0     0    0    0       0                  0    0   
6    鬼子来了   0     0     0     0    0    0       0                  0    0   
7    无问西东   0     0     0     0    0    0       0                  0    0   
8    邪不压正   0     0     0     0    0    0       0                  0    0   
9  11度青春之   0     0     0     0    0    0       0                  0    0   

   ...  龚一飞  龚业珩  龚子涵  龚平  龚格尔  龚洁  龚稼农  龚蓓苾  龚雪  龚黎伟  
0  ...    0    0    0   0    0   0    0    0   0    0  
1  ...    0    0    0   0    0   0    0 

In [6]:
movies_standard.columns

Index(['电影名', '14', 'AYO哟', 'Abby', 'Aron', 'A大调', 'BY2', 'Bailey',
       'Bernardo Loureiro', 'Bin',
       ...
       '龚一飞', '龚业珩', '龚子涵', '龚平', '龚格尔', '龚洁', '龚稼农', '龚蓓苾', '龚雪', '龚黎伟'],
      dtype='object', length=6326)

In [8]:
# 利用mlxtend提供的apriori算法函数得到频繁项集，其中设置最小支持度为0.06
movies_standard.set_index(['电影名'], inplace=True)
frequent_item_sets = apriori(movies_standard, min_support=0.06, use_colnames=True)
print(frequent_item_sets)


D:\soft\python\Lib\site-packages\mlxtend\frequent_patterns\fpcommon.py:113: DeprecationWarning: DataFrames with non-bool types result in worse computationalperformance and their support might be discontinued in the future.Please use a DataFrame with bool type
  DeprecationWarning,


Empty DataFrame
Columns: [support, itemsets]
Index: []


In [21]:
frequent_item_sets = apriori(movies_standard, min_support=0.007, use_colnames=True)
print(frequent_item_sets)


D:\soft\python\Lib\site-packages\mlxtend\frequent_patterns\fpcommon.py:113: DeprecationWarning: DataFrames with non-bool types result in worse computationalperformance and their support might be discontinued in the future.Please use a DataFrame with bool type
  DeprecationWarning,


    support itemsets
0  0.009040     (刘桦)
1  0.007827     (刘烨)
2  0.009811     (姜文)
3  0.007166    (王宝强)
4  0.009371     (葛优)
5  0.008158     (郑恺)
6  0.007276     (郭涛)
7  0.011686     (黄渤)


In [18]:
frequent_item_sets ['length']=frequent_item_sets['itemsets'].apply(lambda x:len(x));frequent_item_sets 


,support,itemsets,length
0,0.003417,(上官云珠),1
1,0.004079,(乔任梁),1
2,0.003748,(井柏然),1
3,0.005733,(余男),1
4,0.003307,(余皑磊),1
...,...,...,...
141,0.003969,(黄璐),1
142,0.003417,(黄觉),1
143,0.003417,(黄轩),1
144,0.003087,"(徐峥, 黄渤)",2


In [20]:
frequent_item_sets [frequent_item_sets ['length']==2]

,support,itemsets,length
144,0.003087,"(徐峥, 黄渤)",2
145,0.003528,"(肖央, 王太利)",2


# 3

In [10]:
# 读取原始数据
movies = pd.read_csv('电影基本信息.csv',encoding='gbk')
movies.head(10)  

,主演,5星,制片国家/地区,编剧,4星,类型,豆瓣评分,3星,上映日期,1星,2星,电影名称,片长
0,刘烨/冯远征/张嘉译/陈坤/马少骅/李沁/周润发/赵本山/黄磊/吕良伟/刘德华/廖凡/董洁/...,NaN,中国大陆 / 香港,董哲/郭俊立/黄建新,NaN,剧情/历史,NaN,NaN,2011/6/15,NaN,NaN,建党伟业,124分钟
1,希亚·拉博夫/罗茜·汉丁顿-惠特莉/乔什·杜哈明/泰瑞斯·吉布森/约翰·马尔科维奇/弗兰西斯...,12.80%,美国,伊伦·克鲁格,36.90%,动作/科幻,7.0,41.10%,2011/7/21,1.70%,7.50%,变形金刚3 Transformers: Dark of the Moon,154分钟
2,杰克·布莱克/安吉丽娜·朱莉/达斯汀·霍夫曼/加里·奥德曼/成龙/塞斯·罗根/刘玉玲/大卫·...,26.70%,美国,乔纳森·阿贝尔/格伦·伯杰,48.30%,喜剧/动作/动画/冒险,8.0,23.00%,2011/5/28,0.30%,1.70%,功夫熊猫2 Kung Fu Panda 2,91分钟
3,克里斯蒂安·贝尔/倪妮/张歆怡/黄天元/韩熙庭/张逗逗/佟大为/曹可凡/渡部笃郎/黄海波/窦...,29.00%,中国大陆 / 香港,刘恒/严歌苓,45.80%,剧情/历史/战争,8.0,21.10%,2011/12/15,1.20%,2.90%,金陵十三钗,145分钟
4,李连杰/周迅/陈坤/李宇春/桂纶镁/范晓萱/樊少皇/杜奕衡/刘家辉/张馨予/盛鉴/薛剑/王双...,9.00%,中国大陆 / 香港,徐克/何冀平/朱雅欐,31.70%,动作/武侠/古装,6.7,46.30%,2011/12/15,2.40%,10.60%,龙门飞甲,120分钟
5,约翰尼·德普/佩内洛普·克鲁兹/杰弗里·拉什/伊恩·麦柯肖恩/山姆·克拉弗林/阿斯特丽德·伯...,17.20%,美国,泰德·艾略特/特里·鲁西奥,44.10%,动作/奇幻/冒险,7.5,34.50%,2011/5/20,0.50%,3.70%,加勒比海盗4：惊涛怪浪 Pirates of the Caribbean: On Stran...,136分钟
6,丹尼尔·雷德克里夫/艾玛·沃森/鲁伯特·格林特/海伦娜·伯翰·卡特/拉尔夫·费因斯/艾伦·瑞...,49.00%,美国 / 英国,史蒂夫·克洛夫斯/J·K·罗琳,36.10%,剧情/悬疑/奇幻/冒险,8.6,13.60%,2011/8/4,0.40%,1.10%,哈利·波特与死亡圣器(下) Harry Potter and the Deathly Hal...,130分钟
7,白百何/文章/张嘉译/王耀庆/张子萱/郭京飞/曹翠芬/魏宗万/海清/廖凡/徐梵溪/李念/焦俊...,15.00%,中国大陆,鲍鲸鲸,42.10%,剧情/爱情,7.3,37.00%,2011/11/8,1.10%,4.90%,失恋33天,110分钟
8,尼尔·帕特里克·哈里斯/杰玛·梅斯/汉克·阿扎利亚/乔纳森·温特斯/凯蒂·派瑞/艾伦·卡明/...,14.90%,美国 / 比利时,J·大卫·斯提姆/戴维·N·韦斯/佩约/杰·舒瑞克/大卫·隆恩,39.20%,喜剧/动画/家庭/奇幻,7.2,38.80%,2011/8/10,1.10%,6.10%,蓝精灵 The Smurfs,103分钟
9,范·迪塞尔/保罗·沃克/道恩·强森/乔丹娜·布鲁斯特/泰瑞斯·吉布森/卢达克里斯/马特·斯查...,38.40%,美国,克里斯·摩根/盖瑞·斯科特·汤普森,44.80%,动作/犯罪,8.4,15.30%,2011/5/12,0.30%,1.20%,速度与激情5 Fast Five,130分钟


In [11]:
mov=movies[['电影名称','类型']]
mov.head()

,电影名称,类型
0,建党伟业,剧情/历史
1,变形金刚3 Transformers: Dark of the Moon,动作/科幻
2,功夫熊猫2 Kung Fu Panda 2,喜剧/动作/动画/冒险
3,金陵十三钗,剧情/历史/战争
4,龙门飞甲,动作/武侠/古装


In [12]:
# 第一步，将原始数据转化为标准格式
movies_standard = mov.drop('类型', axis=1).join(mov['类型'].replace(r'\s*/\s*', '|', regex=True).str.strip().str.get_dummies())

In [13]:
#movies_standard = mov.drop('类型', 1).join(mov['类型'].str.get_dummies())
print(movies_standard.head(10))
# 一共包含 250部电影，一共有29种不同的电影类型（有2列是ID和电影名）
print(movies_standard.shape)  # (250, 30)

                                                电影名称  传记  儿童  冒险  剧情  动作  动画  \
0                                               建党伟业   0   0   0   1   0   0   
1               变形金刚3 Transformers: Dark of the Moon   0   0   0   0   1   0   
2                              功夫熊猫2 Kung Fu Panda 2   0   0   1   0   1   1   
3                                              金陵十三钗   0   0   0   1   0   0   
4                                               龙门飞甲   0   0   0   0   1   0   
5  加勒比海盗4：惊涛怪浪 Pirates of the Caribbean: On Stran...   0   0   1   0   1   0   
6  哈利·波特与死亡圣器(下) Harry Potter and the Deathly Hal...   0   0   1   1   0   0   
7                                              失恋33天   0   0   0   1   0   0   
8                                     蓝精灵 The Smurfs   0   0   0   0   0   1   
9                                   速度与激情5 Fast Five   0   0   0   0   1   0   

   历史  古装  喜剧  ...  恐怖  悬疑  惊悚  战争  武侠  灾难  爱情  犯罪  科幻  运动  
0   1   0   0  ...   0   0   0   0   0   0   0   0   0   0

In [15]:
# 利用mlxtend提供的apriori算法函数得到频繁项集，其中设置最小支持度为0.05
movies_standard.set_index(['电影名称'], inplace=True)
frequent_item_sets = apriori(movies_standard, min_support=0.05, use_colnames=True)
print(frequent_item_sets)

     support      itemsets
0   0.420000          (冒险)
1   0.306667          (剧情)
2   0.620000          (动作)
3   0.073333          (动画)
4   0.053333          (古装)
5   0.280000          (喜剧)
6   0.200000          (奇幻)
7   0.086667          (悬疑)
8   0.106667          (惊悚)
9   0.153333          (爱情)
10  0.126667          (犯罪)
11  0.246667          (科幻)
12  0.053333      (冒险, 剧情)
13  0.306667      (冒险, 动作)
14  0.066667      (喜剧, 冒险)
15  0.140000      (冒险, 奇幻)
16  0.053333      (冒险, 惊悚)
17  0.173333      (冒险, 科幻)
18  0.126667      (剧情, 动作)
19  0.073333      (剧情, 爱情)
20  0.060000      (犯罪, 剧情)
21  0.093333      (喜剧, 动作)
22  0.126667      (奇幻, 动作)
23  0.073333      (动作, 惊悚)
24  0.120000      (犯罪, 动作)
25  0.213333      (科幻, 动作)
26  0.066667      (喜剧, 动画)
27  0.080000      (喜剧, 爱情)
28  0.093333  (冒险, 奇幻, 动作)
29  0.146667  (冒险, 科幻, 动作)
30  0.053333  (犯罪, 剧情, 动作)


D:\soft\python\Lib\site-packages\mlxtend\frequent_patterns\fpcommon.py:113: DeprecationWarning: DataFrames with non-bool types result in worse computationalperformance and their support might be discontinued in the future.Please use a DataFrame with bool type
  DeprecationWarning,


In [16]:
# 计算规则，并设置提升度阈值为 1.25 （返回的是各个指标的数值，可以按照按兴趣的指标排序观察，但具体解释还得参考实际数据的含义）
rules = association_rules(frequent_item_sets, metric='lift', min_threshold=1.25)
rules

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,leverage,conviction,zhangs_metric
0,(冒险),(奇幻),0.420000,0.200000,0.140000,0.333333,1.666667,0.056000,1.200000,0.689655
1,(奇幻),(冒险),0.200000,0.420000,0.140000,0.700000,1.666667,0.056000,1.933333,0.500000
2,(冒险),(科幻),0.420000,0.246667,0.173333,0.412698,1.673102,0.069733,1.282703,0.693634
3,(科幻),(冒险),0.246667,0.420000,0.173333,0.702703,1.673102,0.069733,1.950909,0.534037
4,(剧情),(爱情),0.306667,0.153333,0.073333,0.239130,1.559546,0.026311,1.112762,0.517483
5,(爱情),(剧情),0.153333,0.306667,0.073333,0.478261,1.559546,0.026311,1.328889,0.423765
6,(犯罪),(剧情),0.126667,0.306667,0.060000,0.473684,1.544622,0.021156,1.317333,0.403732
7,(剧情),(犯罪),0.306667,0.126667,0.060000,0.195652,1.544622,0.021156,1.085766,0.508547
8,(犯罪),(动作),0.126667,0.620000,0.120000,0.947368,1.528014,0.041467,7.220000,0.395674
9,(动作),(犯罪),0.620000,0.126667,0.120000,0.193548,1.528014,0.041467,1.082933,0.909357


In [17]:
# 对lift降序排序，查看lift较大的是哪些规则
rules_sort = rules.sort_values(by=['lift'], ascending=False)
rules_sort

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,leverage,conviction,zhangs_metric
28,"(剧情, 动作)",(犯罪),0.126667,0.126667,0.053333,0.421053,3.324100,0.037289,1.508485,0.800573
29,(犯罪),"(剧情, 动作)",0.126667,0.126667,0.053333,0.421053,3.324100,0.037289,1.508485,0.800573
12,(喜剧),(动画),0.280000,0.073333,0.066667,0.238095,3.246753,0.046133,1.216250,0.961111
13,(动画),(喜剧),0.073333,0.280000,0.066667,0.909091,3.246753,0.046133,7.920000,0.746763
24,(科幻),"(冒险, 动作)",0.246667,0.306667,0.146667,0.594595,1.938895,0.071022,1.710222,0.642800
21,"(冒险, 动作)",(科幻),0.306667,0.246667,0.146667,0.478261,1.938895,0.071022,1.443889,0.698427
15,(爱情),(喜剧),0.153333,0.280000,0.080000,0.521739,1.863354,0.037067,1.505455,0.547244
14,(喜剧),(爱情),0.280000,0.153333,0.080000,0.285714,1.863354,0.037067,1.185333,0.643519
17,"(奇幻, 动作)",(冒险),0.126667,0.420000,0.093333,0.736842,1.754386,0.040133,2.204000,0.492366
18,(冒险),"(奇幻, 动作)",0.420000,0.126667,0.093333,0.222222,1.754386,0.040133,1.122857,0.741379


In [20]:
# 计算规则，并设置提升度阈值为 1.25 （返回的是各个指标的数值，可以按照按兴趣的指标排序观察，但具体解释还得参考实际数据的含义）
rules = association_rules(frequent_item_sets, metric='confidence', min_threshold=0.9)
rules

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,leverage,conviction,zhangs_metric
0,(犯罪),(动作),0.126667,0.62,0.120000,0.947368,1.528014,0.041467,7.22,0.395674
1,(动画),(喜剧),0.073333,0.28,0.066667,0.909091,3.246753,0.046133,7.92,0.746763


In [23]:
# 读取原始数据
movies = pd.read_excel('aiqiyi.xlsx')
movies.head(10)  

,演员,类型,整理后剧名,上映时间,语言,评分,地区,导演,差评数,评分人数,播放量,好评数
0,ELLA/吴尊/汪东城/唐禹哲/唐治平/阮经天/张皓明/龚继安/郭静纯,言情剧/偶像剧/青春剧/粤语电视剧/超清1080P,花样少年少女,2006,国语,8.8,台湾,林清芳/王明台,5498,45800,26090000,40302
1,UEE/崔宇植/任瑟雍,言情剧/剧情/都市/粤语电视剧/超清1080P,浩9的爱情,2015,韩语,8.0,韩国,表民洙,2065,10265,6140000,8200
2,Weir Sukollwat/Usamanee Vaithayanon/查亚鹏·普帕特,偶像剧/言情剧/剧情/年代剧/粤语电视剧/超清1080P,命定之爱国语,2007,国语,8.5,泰国,未知,5458,35846,38190000,30388
3,丁勇岱,罪案剧/剧情/粤语电视剧/超清1080P,末路1997,2000,国语,7.5,内地,陈国军,963,3832,10360000,2869
4,丁勇岱/刘威葳/郭涛/周晓鸥,剧情/都市/粤语电视剧,暗伤,2006,国语,6.0,内地,高群书,279,691,2150000,412
5,丁勇岱/刘欣,剧情/都市/悬疑剧/粤语电视剧,威胁,2003,国语,5.3,内地,陈国军,358,759,1850000,401
6,丁勇岱/梁静/刘金山/海一天,罪案剧/剧情/都市/粤语电视剧,水晶眼,2006,国语,6.0,内地,常猛,223,554,1240000,331
7,丁勇岱/谢兰,言情剧/剧情/年代剧/粤语电视剧/超清1080P,驼道,1995,国语,6.2,内地,史晨风,104,275,519000,171
8,丁勇岱/陈瑾/曹克难/石维坚/王诗槐/宋春丽,罪案剧/剧情/都市/粤语电视剧,不堪回首,2005,国语,5.8,内地,唐敬睿,73,172,433000,99
9,丁志成/张彤/王海地/沈丹萍/赵曦/尹国华,罪案剧/剧情/粤语电视剧,凝视黑夜,2001,国语,5.8,内地,陈胜利,397,936,2130000,539


In [25]:
mov=movies[['整理后剧名','类型']]
mov.head()

,整理后剧名,类型
0,花样少年少女,言情剧/偶像剧/青春剧/粤语电视剧/超清1080P
1,浩9的爱情,言情剧/剧情/都市/粤语电视剧/超清1080P
2,命定之爱国语,偶像剧/言情剧/剧情/年代剧/粤语电视剧/超清1080P
3,末路1997,罪案剧/剧情/粤语电视剧/超清1080P
4,暗伤,剧情/都市/粤语电视剧


In [26]:
# 第一步，将原始数据转化为标准格式
movies_standard = mov.drop('类型', axis=1).join(mov['类型'].replace(r'\s*/\s*', '|', regex=True).str.strip().str.get_dummies())

In [27]:
#movies_standard = mov.drop('类型', 1).join(mov['类型'].str.get_dummies())
print(movies_standard.head(10))
# 一共包含 250部电影，一共有29种不同的电影类型（有2列是ID和电影名）
print(movies_standard.shape)  # (250, 30)

    整理后剧名  偶像剧  军旅剧  农村剧  剧情  历史剧  古装剧  商战剧  喜剧  宫廷剧  ...  科幻剧  穿越剧  粤语电视剧  \
0  花样少年少女    1    0    0   0    0    0    0   0    0  ...    0    0      1   
1   浩9的爱情    0    0    0   1    0    0    0   0    0  ...    0    0      1   
2  命定之爱国语    1    0    0   1    0    0    0   0    0  ...    0    0      1   
3  末路1997    0    0    0   1    0    0    0   0    0  ...    0    0      1   
4      暗伤    0    0    0   1    0    0    0   0    0  ...    0    0      1   
5      威胁    0    0    0   1    0    0    0   0    0  ...    0    0      1   
6     水晶眼    0    0    0   1    0    0    0   0    0  ...    0    0      1   
7      驼道    0    0    0   1    0    0    0   0    0  ...    0    0      1   
8    不堪回首    0    0    0   1    0    0    0   0    0  ...    0    0      1   
9    凝视黑夜    0    0    0   1    0    0    0   0    0  ...    0    0      1   

   网络剧  罪案剧  言情剧  谍战剧  超清1080P  都市  青春剧  
0    0    0    1    0        1   0    1  
1    0    0    1    0        1   1    0  
2    0    0    

In [28]:
# 利用mlxtend提供的apriori算法函数得到频繁项集，其中设置最小支持度为0.05
movies_standard.set_index(['整理后剧名'], inplace=True)
frequent_item_sets = apriori(movies_standard, min_support=0.05, use_colnames=True)
print(frequent_item_sets)

      support                    itemsets
0    0.102772                       (偶像剧)
1    0.103448                       (军旅剧)
2    0.551048                        (剧情)
3    0.098039                       (历史剧)
4    0.144692                       (古装剧)
..        ...                         ...
114  0.088573    (都市, 粤语电视剧, 超清1080P, 剧情)
115  0.085193  (言情剧, 粤语电视剧, 超清1080P, 家庭剧)
116  0.053414  (言情剧, 粤语电视剧, 超清1080P, 年代剧)
117  0.079784   (言情剧, 都市, 粤语电视剧, 超清1080P)
118  0.076403  (言情剧, 粤语电视剧, 超清1080P, 青春剧)

[119 rows x 2 columns]


D:\soft\python\Lib\site-packages\mlxtend\frequent_patterns\fpcommon.py:113: DeprecationWarning: DataFrames with non-bool types result in worse computationalperformance and their support might be discontinued in the future.Please use a DataFrame with bool type
  DeprecationWarning,


In [30]:
frequent_item_sets.sort_values(by='support',ascending=False)

,support,itemsets
9,0.954023,(粤语电视剧)
53,0.639621,"(粤语电视剧, 超清1080P)"
13,0.639621,(超清1080P)
2,0.551048,(剧情)
28,0.523327,"(粤语电视剧, 剧情)"
...,...,...
96,0.053414,"(言情剧, 超清1080P, 年代剧)"
116,0.053414,"(言情剧, 粤语电视剧, 超清1080P, 年代剧)"
48,0.052738,"(悬疑剧, 罪案剧)"
97,0.050034,"(悬疑剧, 罪案剧, 粤语电视剧)"


In [29]:
# 计算规则，并设置提升度阈值为 1.25 （返回的是各个指标的数值，可以按照按兴趣的指标排序观察，但具体解释还得参考实际数据的含义）
rules = association_rules(frequent_item_sets, metric='lift', min_threshold=1.25)
rules

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,leverage,conviction,zhangs_metric
0,(言情剧),(偶像剧),0.432725,0.102772,0.087221,0.201563,1.961256,0.042749,1.123730,0.863995
1,(偶像剧),(言情剧),0.102772,0.432725,0.087221,0.848684,1.961256,0.042749,3.748949,0.546263
2,(青春剧),(偶像剧),0.146721,0.102772,0.060176,0.410138,3.990753,0.045097,1.521082,0.878283
3,(偶像剧),(青春剧),0.102772,0.146721,0.060176,0.585526,3.990753,0.045097,2.058705,0.835262
4,(军旅剧),(超清1080P),0.103448,0.639621,0.089249,0.862745,1.348837,0.023082,2.625616,0.288462
...,...,...,...,...,...,...,...,...,...,...
103,"(言情剧, 超清1080P)","(粤语电视剧, 青春剧)",0.283976,0.137931,0.076403,0.269048,1.950595,0.037234,1.179378,0.680614
104,"(粤语电视剧, 青春剧)","(言情剧, 超清1080P)",0.137931,0.283976,0.076403,0.553922,1.950595,0.037234,1.605153,0.565310
105,"(青春剧, 超清1080P)","(言情剧, 粤语电视剧)",0.105477,0.405003,0.076403,0.724359,1.788526,0.033685,2.158592,0.492866
106,(言情剧),"(粤语电视剧, 超清1080P, 青春剧)",0.432725,0.105477,0.076403,0.176563,1.673948,0.030761,1.086328,0.709726


In [32]:
# 计算规则，并设置提升度阈值为 1.25 （返回的是各个指标的数值，可以按照按兴趣的指标排序观察，但具体解释还得参考实际数据的含义）
rules = association_rules(frequent_item_sets, metric='confidence', min_threshold=0.6)
rules

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,leverage,conviction,zhangs_metric
0,(偶像剧),(粤语电视剧),0.102772,0.954023,0.098715,0.960526,1.006817,0.000668,1.164751,0.007546
1,(偶像剧),(言情剧),0.102772,0.432725,0.087221,0.848684,1.961256,0.042749,3.748949,0.546263
2,(偶像剧),(超清1080P),0.102772,0.639621,0.078431,0.763158,1.193140,0.012696,1.521599,0.180417
3,(军旅剧),(粤语电视剧),0.103448,0.954023,0.103448,1.000000,1.048193,0.004756,inf,0.051282
4,(军旅剧),(超清1080P),0.103448,0.639621,0.089249,0.862745,1.348837,0.023082,2.625616,0.288462
...,...,...,...,...,...,...,...,...,...,...
135,"(言情剧, 粤语电视剧, 青春剧)",(超清1080P),0.098039,0.639621,0.076403,0.779310,1.218393,0.013695,1.632966,0.198730
136,"(言情剧, 青春剧, 超清1080P)",(粤语电视剧),0.076403,0.954023,0.076403,1.000000,1.048193,0.003513,inf,0.049780
137,"(粤语电视剧, 超清1080P, 青春剧)",(言情剧),0.105477,0.432725,0.076403,0.724359,1.673948,0.030761,2.058022,0.450083
138,"(言情剧, 青春剧)","(粤语电视剧, 超清1080P)",0.104801,0.639621,0.076403,0.729032,1.139787,0.009370,1.329969,0.137001


In [35]:
rules[rules['confidence']==1].shape

(28, 10)

In [36]:
rules[rules['confidence']==1]

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,leverage,conviction,zhangs_metric
3,(军旅剧),(粤语电视剧),0.103448,0.954023,0.103448,1.0,1.048193,0.004756,inf,0.051282
22,(谍战剧),(粤语电视剧),0.080460,0.954023,0.080460,1.0,1.048193,0.003699,inf,0.050000
24,(超清1080P),(粤语电视剧),0.639621,0.954023,0.639621,1.0,1.048193,0.029408,inf,0.127580
35,"(超清1080P, 偶像剧)",(粤语电视剧),0.078431,0.954023,0.078431,1.0,1.048193,0.003606,inf,0.049890
43,"(军旅剧, 剧情)",(粤语电视剧),0.055443,0.954023,0.055443,1.0,1.048193,0.002549,inf,0.048676
45,"(军旅剧, 超清1080P)",(粤语电视剧),0.089249,0.954023,0.089249,1.0,1.048193,0.004103,inf,0.050483
61,"(剧情, 超清1080P)",(粤语电视剧),0.334686,0.954023,0.334686,1.0,1.048193,0.015388,inf,0.069106
68,"(历史剧, 超清1080P)",(粤语电视剧),0.067613,0.954023,0.067613,1.0,1.048193,0.003109,inf,0.049311
72,"(超清1080P, 古装剧)",(粤语电视剧),0.094659,0.954023,0.094659,1.0,1.048193,0.004352,inf,0.050784
75,"(喜剧, 超清1080P)",(粤语电视剧),0.072346,0.954023,0.072346,1.0,1.048193,0.003326,inf,0.049563
